In [13]:
# Cell 1 — check GPU
!nvidia-smi

Fri Oct  2 06:00:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [14]:
# Cell 2 — install pinned deps (keep Kaggle's CUDA torch build; the actual torch version is recorded in config.json)
import glob, os
IN = os.path.dirname(glob.glob("/kaggle/input/**/train_simcse.py", recursive=True)[0])
print("dataset dir:", IN)
!grep -v '^torch==' {IN}/requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt
import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.is_available())

dataset dir: /kaggle/input/datasets/juliodeaquino/simcse-snli-inputs
2.10.0+cu128 5.18.0 True


In [15]:
!ls -la /kaggle/input/datasets/juliodeaquino/simcse-snli-inputs/

total 8480
drwxr-xr-x 2 nobody nogroup       0 Oct  2 06:00 .
drwxr-xr-x 3 root   root       4096 Oct  2 06:00 ..
-rw-r--r-- 1 nobody nogroup   10487 Oct  2 06:00 evaluate.py
-rw-r--r-- 1 nobody nogroup     107 Oct  2 06:00 requirements.txt
-rw-r--r-- 1 nobody nogroup     494 Oct  2 06:00 SHA256SUMS
-rw-r--r-- 1 nobody nogroup  266898 Oct  2 06:00 stsb_dev.jsonl
-rw-r--r-- 1 nobody nogroup   16253 Oct  2 06:00 train_simcse.py
-rw-r--r-- 1 nobody nogroup     712 Oct  2 06:00 unsupervised.json
-rw-r--r-- 1 nobody nogroup 8366751 Oct  2 06:00 unsup_sentences.txt


In [16]:
# Cell 3 — recreate the repo layout under /kaggle/working so paths resolve unmodified
import shutil
W = "/kaggle/working"
layout = {"train_simcse.py": "src/train_simcse.py", "evaluate.py": "src/evaluate.py",
          "unsupervised.json": "configs/unsupervised.json",
          "unsup_sentences.txt": "data/processed/unsup_sentences.txt",
          "stsb_dev.jsonl": "data/stsb/dev.jsonl", "requirements.txt": "requirements.txt"}
for src, dst in layout.items():
    os.makedirs(os.path.join(W, os.path.dirname(dst)), exist_ok=True)
    shutil.copy(os.path.join(IN, src), os.path.join(W, dst))
os.chdir(W)
!find src configs data -type f | sort

configs/unsupervised.json
data/processed/unsup_sentences.txt
data/stsb/dev.jsonl
src/evaluate.py
src/train_simcse.py


In [17]:
# Cell 4 — train (Mode A, unsupervised, Eq. 1)
RUN_ID = "unsup_seed42_bs64"
!cd /kaggle/working && python src/train_simcse.py --mode unsup --config configs/unsupervised.json --output_dir runs/{RUN_ID} --hardware "Kaggle T4 x2" 

config.json: 100%|█████████████████████████████| 570/570 [00:00<00:00, 2.11MB/s]
tokenizer_config.json: 100%|██████████████████| 48.0/48.0 [00:00<00:00, 134kB/s]
vocab.txt: 100%|█████████████████████████████| 232k/232k [00:00<00:00, 9.49MB/s]
tokenizer.json: 100%|████████████████████████| 466k/466k [00:00<00:00, 15.0MB/s]

model.safetensors: downloading bytes: ██████████            |  201MB, 14.3MB/s  
model.safetensors: downloading bytes: ██████████████▍       |  289MB, 23.7MB/s  
model.safetensors: downloading bytes: ██████████████████▉   |  379MB, 31.5MB/s  
model.safetensors: downloading bytes: ██████████████████████|  415MB, 38.6MB/s  
model.safetensors: reconstructing file: 100%|██████|  440MB /  440MB, 42.0MB/s  
Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 2434.86it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.

In [18]:
# Cell 5 — verify run_record.json (written by the script) and trim the output
import json
rec = json.load(open(f"/kaggle/working/runs/{RUN_ID}/run_record.json"))
need = {"run_id","mode","timestamp","config","seed","hardware","results","checkpoint_path","notes"}
assert need <= rec.keys(), need - rec.keys()
assert rec["results"]["dev_spearman"] is not None and rec["results"]["test_spearman"] is None
print(json.dumps({k: rec[k] for k in ("run_id","hardware","results","notes")}, indent=2))
shutil.rmtree("/kaggle/working/data")   # inputs, no need to re-download them
!find /kaggle/working/runs -type f | sort && du -sh /kaggle/working/runs

{
  "run_id": "unsup_seed42_bs64",
  "hardware": {
    "device": "Kaggle T4 x2",
    "num_gpus": 1,
    "num_gpus_available": 2,
    "precision": "fp16 (amp)",
    "gpu_name": "Tesla T4"
  },
  "results": {
    "dev_spearman": 0.7692467983004365,
    "test_spearman": null,
    "best_step": 125
  },
  "notes": "final training loss < 1e-3: possible collapse (identical views?)"
}
/kaggle/working/runs/unsup_seed42_bs64/checkpoint/config.json
/kaggle/working/runs/unsup_seed42_bs64/checkpoint/model.safetensors
/kaggle/working/runs/unsup_seed42_bs64/checkpoint/tokenizer_config.json
/kaggle/working/runs/unsup_seed42_bs64/checkpoint/tokenizer.json
/kaggle/working/runs/unsup_seed42_bs64/config.json
/kaggle/working/runs/unsup_seed42_bs64/results.json
/kaggle/working/runs/unsup_seed42_bs64/run_record.json
419M	/kaggle/working/runs
